In [1]:
import pandas as pd
from sklearn.ensemble import IsolationForest


In [2]:
# load data
df = pd.read_csv("cleaned_expenses.csv")


In [3]:

# convert date
df['date'] = pd.to_datetime(df['date'])

In [4]:

# initialize anomaly column
df['anomaly'] = 0

In [5]:
# loop through each category
for cat in df['category'].unique():
    
    # filter category data
    temp = df[df['category'] == cat].copy()
    
    # skip small datasets
    if len(temp) < 10:
        continue
    
    # select feature
    X = temp[['amount']]
    
    # train isolation forest
    model = IsolationForest(contamination=0.05, random_state=42)
    preds = model.fit_predict(X)
    
    # convert labels to 0/1
    preds = pd.Series(preds, index=temp.index).map({1: 0, -1: 1})
    
    # assign back to main dataframe
    df.loc[temp.index, 'anomaly'] = preds

In [6]:
# extract anomalies
anomalies = df[df['anomaly'] == 1]


In [7]:
# print anomalies
print("Detected anomalies:\n")
print(anomalies[['date', 'amount', 'category', 'title']])


Detected anomalies:

                       date   amount       category           title
17  2024-07-15 08:50:27.789   3800.0   bills & fees            rent
33  2024-06-30 15:25:49.013    520.0  food & drinks         biryani
37  2024-06-29 17:30:35.612    180.0      transport             cab
45  2024-06-28 01:21:47.390    358.0    not defined  adjust balance
58  2024-06-26 16:20:10.049    200.0      transport             cab
73  2024-06-17 14:56:11.086    260.0      transport         tickets
78  2024-06-16 08:17:00.000    170.0      transport             cab
86  2024-06-13 15:22:05.293    200.0      transport          petrol
89  2024-06-09 05:26:00.000    200.0      transport          petrol
107 2024-05-31 03:09:00.000    520.0  food & drinks         biryani
182 2024-05-04 03:27:00.000    200.0      transport          petrol
194 2024-05-02 01:46:00.656   5000.0   bills & fees            baba
233 2024-04-23 17:13:53.508    795.0  food & drinks      team lunch
239 2024-04-22 14:41:00.000

In [8]:
# save results
df.to_csv("anomaly_results.csv", index=False)